# Create Train-Test-Validate Splits

In [5]:
import pandas as pd
from sklearn.utils import shuffle

In [38]:
def load_data(
    dir: str,
    split: list = [60, 20, 20],
    seed: int = 12345
    ):

    final_output = {}

    for dataset in ['affectivetext', 'crowdflower', 'electoraltweets', 'ssec', 'emoint', 'goemotions', 'tec']:
        dataset_output = {}

        # Load Data
        df_original = pd.read_parquet(f'{dir}//original//{dataset}.parquet')
        df_primary = pd.read_parquet(f'{dir}//primary//{dataset}.parquet')
        df_secondary = pd.read_parquet(f'{dir}//secondary//{dataset}.parquet')

        def splits(data):
            # Output dict
            output = {}

            # Randomise
            df = shuffle(data, random_state=seed).reset_index(drop=True)

            # Split datasets - original split
            if len(df['dataset'].unique()) != 1:
                if len(df['dataset'].unique()) == 3:
                    train = df.loc[df['dataset'] == 'train', :]
                    test = df.loc[df['dataset'] == 'test', :]
                    val = df.loc[df['dataset'] == 'validation', :]
                else:
                    train = df.loc[df['dataset'] == 'train', :]
                    test = df.loc[df['dataset'] == 'test', :]
                    val = df.loc[df['dataset'] == 'test', :]

                train = train.assign(dataset = 'train')
                test = test.assign(dataset = 'test')
                val = val.assign(dataset = 'val')

                print(f"|---- DEFAULT SPLIT = Train: {len(train)} | Test: {len(test)} | Validation: {len(val)}")

                output['default'] = pd.concat([train, test, val])

            # Split datasets - custom split
            splits = [int(round(sum(split[:i+1])/100 * len(df), 0)) for i,n in enumerate(split)]
            if len(df) % 3:
                splits[0] += 1
            train = df.loc[:splits[0]+1, :]
            test = df.loc[splits[0]:splits[1]+1, :]
            val = df.loc[splits[1]:, :]

            train = train.assign(dataset = 'train')
            test = test.assign(dataset = 'test')
            val = val.assign(dataset = 'val')

            print(f"|---- {'-'.join([str(i) for i in split])} SPLIT = Train:{len(train)} | Test: {len(test)} | Validation: {len(val)}")

            output['custom_split'] = pd.concat([train, test, val])

            return output

        print(f"{dataset}: {len(df_original)}")
        print(f"|-- ORIGINAL: {len(df_original)}")
        dataset_output['original'] = splits(df_original)

        print(f"|-- PRIMARY: {len(df_primary)}")
        dataset_output['primary'] = splits(df_primary)

        print(f"|-- SECONDARY: {len(df_secondary)}")
        dataset_output['secondary'] = splits(df_secondary)
        print("")

        final_output[dataset] = dataset_output

    return final_output

In [39]:
test = load_data('data//clean')

affectivetext: 10690
|-- ORIGINAL: 10690
|---- 60-20-20 SPLIT = Train:6417 | Test: 2139 | Validation: 2138
|-- PRIMARY: 10248
|---- 60-20-20 SPLIT = Train:6151 | Test: 2051 | Validation: 2050
|-- SECONDARY: 10690
|---- 60-20-20 SPLIT = Train:6417 | Test: 2139 | Validation: 2138

crowdflower: 30535
|-- ORIGINAL: 30535
|---- 60-20-20 SPLIT = Train:18324 | Test: 6108 | Validation: 6107
|-- PRIMARY: 12671
|---- 60-20-20 SPLIT = Train:7606 | Test: 2535 | Validation: 2534
|-- SECONDARY: 30535
|---- 60-20-20 SPLIT = Train:18324 | Test: 6108 | Validation: 6107

electoraltweets: 4055
|-- ORIGINAL: 4055
|---- 60-20-20 SPLIT = Train:2436 | Test: 812 | Validation: 811
|-- PRIMARY: 1362
|---- 60-20-20 SPLIT = Train:819 | Test: 275 | Validation: 272
|-- SECONDARY: 3935
|---- 60-20-20 SPLIT = Train:2364 | Test: 788 | Validation: 787

ssec: 1534
|-- ORIGINAL: 1534
|---- DEFAULT SPLIT = Train: 921 | Test: 613 | Validation: 613
|---- 60-20-20 SPLIT = Train:923 | Test: 308 | Validation: 307
|-- PRIMARY: 